# M4 — Train

**Trạng thái:** `FROZEN_OUTPUT_VERIFIED`  
**Chế độ mặc định:** CPU · read-only · không train · không resume · không chạy test inference.  

Bốn nhánh (`original_gan`, `matched_ros`, `random_local`, `learned_local`) × seeds 42/43/44. Mỗi job có ngân sách 8.000 C updates.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Worker

Hash worker đã đối chiếu với source trong hồ sơ M4. Không import hoặc gọi worker.

In [3]:
from src.audit import digest, M4_SHA
p=ROOT/'src/frozen/m4.py'
actual=digest(p)
assert actual==M4_SHA, f'Worker changed: {actual}'
print('M4 worker locked:', actual)

M4 worker locked: 6d33ca6a0d39600f5f44c0d7e63547b1237149d625471ac73315d982ffbd8fb5


## Đầu vào train

Đọc contract trong ZIP gốc khi được cung cấp; không giải nén weights lớn.

In [4]:
from src.audit import verified_m4_registry
observed=verified_m4_registry(INPUT_CONFIG['m4_zips'])
print('Verified final jobs:',len(observed),'| expected: 12')
for row in observed: print(row)

Verified final jobs: 12 | expected: 12
{'arm': 'learned_local', 'seed': 42, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'learned_local', 'seed': 43, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'learned_local', 'seed': 44, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'matched_ros', 'seed': 42, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài

## Registry

Không khẳng định đủ job bằng tên file. Cần metadata/hash và bản ghi hoàn tất của từng job.

In [5]:
for item in observed[:20]:print(item)
print('Source: src/frozen/m4.py, train_launcher.py, group_launcher.py')

{'arm': 'learned_local', 'seed': 42, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'learned_local', 'seed': 43, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'learned_local', 'seed': 44, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\checkpoints\\02_M4_MODELS_12.zip'}
{'arm': 'matched_ros', 'seed': 42, 'stage': 'final', 'status': 'COMPLETE', 'c_updates': 8000, 'adam_clock': 136000, 'protocol': '50e988fae260', 'archive': 'C:\\Users\\ADMIN\\OneDrive\\Tài liệu\\ChatGPT\\KLTN\\KLTN_clean\\check

## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [6]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS
